Summary:
Module uses Azure document AI (formrecognizer) to scan an image embedded in pdf file and extract the text content from the file. 

Specifications : 
* Input files are stored in the directory C:\Users\shsenthi\OneDrive - Microsoft\OfficeStore\Downloads\receipts
* Output JSON files stored in C:\Users\shsenthi\Downloads\ocr_output

Steps :
* login into azure subscription - using web portal 
* get the keys and look for resource group with name pdf and form recongizer with name pdf-form-recognizer-c3160796
* The access keys for this resource is stored in c:\downloads\pdf-resource-key

* Load the input file directory and for each PDF pass the file as input to the service and get JSON output 
* Store the JSON output 



In [ ]:
import os
import json
from azure.ai.formrecognizer import DocumentAnalysisClient
from azure.core.credentials import AzureKeyCredential

# --- Configuration ---
MODEL_ID = "prebuilt-receipt"  # Optimized for receipts: extracts merchant, date, items, totals, tax, etc.
INPUT_DIR = os.path.join("C:", os.sep, "Users", "shsenthi", "OneDrive - Microsoft", "OfficeStore", "Downloads", "receipts")
OUTPUT_DIR = os.path.join("C:", os.sep, "Users", "shsenthi", "Downloads", "ocr_output")
KEY_FILE = os.path.join("C:", os.sep,  "Users", "shsenthi","downloads", "pdf-azure-keys.md")  # File containing endpoint and API key

# Read the API key and endpoint from the key file
with open(KEY_FILE, "r") as f:
    lines = [line.strip() for line in f.readlines() if line.strip()]
    endpoint = lines[0]   # first line: endpoint URL
    api_key = lines[1]    # second line: API key

# Validate endpoint and API key format before making any API calls
if not endpoint or not endpoint.startswith("https://"):
    raise ValueError(f"Invalid endpoint: '{endpoint}'. It should be a valid HTTPS URL.")
if not api_key or len(api_key) < 10:
    raise ValueError("Invalid API key: key appears to be missing or too short.")

print(f"Endpoint : {endpoint}")
print(f"API Key  : {api_key[:6]}{'*' * (len(api_key) - 6)}")
print(f"Model    : {MODEL_ID}")

# Create the Document Analysis client
client = DocumentAnalysisClient(endpoint=endpoint, credential=AzureKeyCredential(api_key))

# Validate credentials by analyzing the first available PDF in the input directory
try:
    test_files = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(".pdf")]
    if not test_files:
        raise FileNotFoundError(f"No PDF files found in {INPUT_DIR} to validate credentials.")
    
    test_path = os.path.join(INPUT_DIR, test_files[0])
    with open(test_path, "rb") as test_pdf:
        test_poller = client.begin_analyze_document(MODEL_ID, document=test_pdf)
        test_result = test_poller.result()
    
    print(f"\nCredentials validated successfully using: {test_files[0]}")
    print(f"  Receipts detected: {len(test_result.documents)}")
except Exception as e:
    raise RuntimeError(f"Failed to validate Azure credentials or process PDF.\nError: {e}")

# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"\nOutput directory ready: {OUTPUT_DIR}")

In [ ]:
import datetime

MAX_FILE_SIZE_MB = 2  # Skip files larger than this size (in MB)
MAX_FILE_SIZE_BYTES = MAX_FILE_SIZE_MB * 1024 * 1024

def make_serializable(obj):
    """Recursively convert an object into a JSON-serializable structure."""
    if obj is None or isinstance(obj, (str, int, float, bool)):
        return obj
    if isinstance(obj, (datetime.date, datetime.time, datetime.datetime)):
        return obj.isoformat()
    if isinstance(obj, dict):
        return {k: make_serializable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [make_serializable(item) for item in obj]
    if hasattr(obj, "to_dict"):
        return make_serializable(obj.to_dict())
    if hasattr(obj, "__dict__"):
        return make_serializable(obj.__dict__)
    return str(obj)

def extract_field(doc, field_name):
    """Extract a field value from a receipt document, returning None if not found."""
    field = doc.fields.get(field_name)
    if field is None:
        return None
    if field.value_type == "currency":
        return {"amount": field.value.amount, "currency": field.value.symbol} if field.value else None
    if field.value_type == "date":
        return str(field.value) if field.value else None
    if field.value_type == "list":
        items = []
        for item in field.value:
            if item.value_type == "dictionary":
                item_dict = {}
                for k, v in item.value.items():
                    if v.value_type == "currency":
                        item_dict[k] = {"amount": v.value.amount, "currency": v.value.symbol} if v.value else None
                    elif v.value_type == "float":
                        item_dict[k] = v.value
                    else:
                        item_dict[k] = v.content
                items.append(item_dict)
        return items
    return field.content if field.content else str(field.value)

# Process each PDF in the input directory
pdf_files = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(".pdf")]
print(f"Found {len(pdf_files)} PDF file(s) in input directory.\n")

# Process only selected files (if list is non-empty), otherwise process all
#scan_files = ["BCR-HYD-527672.pdf"]
scan_files = []  # Empty list means process all files
for filename in pdf_files:
    if scan_files and filename not in scan_files:
        continue

    filepath = os.path.join(INPUT_DIR, filename)

    # Skip files larger than MAX_FILE_SIZE_MB
    file_size = os.path.getsize(filepath)
    if file_size > MAX_FILE_SIZE_BYTES:
        print(f"Skipping: {filename} ({file_size / (1024 * 1024):.2f} MB > {MAX_FILE_SIZE_MB} MB limit)")
        continue

    print(f"Processing: {filename} ({file_size / (1024 * 1024):.2f} MB) ...")

    with open(filepath, "rb") as pdf:
        poller = client.begin_analyze_document(MODEL_ID, document=pdf)
        result = poller.result()

    # Convert to dict, then recursively sanitize all non-serializable values
    result_dict = make_serializable(result.to_dict())

    # Save the full result as JSON
    output_filename = os.path.splitext(filename)[0] + ".json"
    output_path = os.path.join(OUTPUT_DIR, output_filename)
    with open(output_path, "w", encoding="utf-8") as out_f:
        json.dump(result_dict, out_f, indent=2, ensure_ascii=False)

    # Print summary
    doc_count = len(result_dict.get("documents", []))
    print(f"  -> Receipts found: {doc_count}")
    for i, doc in enumerate(result.documents, 1):
        merchant = extract_field(doc, "MerchantName") or "Unknown"
        date = extract_field(doc, "TransactionDate") or "N/A"
        total = extract_field(doc, "Total")
        print(f"     Receipt {i}: {merchant} | {date} | Total: {total}")
    print(f"  -> Saved: {output_filename}")

print("\nDone. All files processed.")

In [ ]:
if result.documents:
    doc = result.documents[0]
    print("Available fields:")
    for field_name in doc.fields.keys():
        print(f"  - {field_name}")


## SLM Entity Extraction using Azure AI

### Steps to initialize an SLM in Azure Portal:

1. **Go to Azure AI Foundry** (https://ai.azure.com) or **Azure Portal**
2. **Create an Azure OpenAI resource** (if not already created):
   - Search for "Azure OpenAI" in the portal
   - Click **Create** → choose your subscription, resource group, region
   - Pricing tier: Standard S0
3. **Deploy a Small Language Model (SLM)**:
   - Open the resource → go to **Azure AI Foundry portal**
   - Navigate to **Deployments** → **+ Create deployment**
   - Choose an SLM like **Phi-3-mini**, **Phi-3.5-mini**, or **Phi-4-mini**
   - Set deployment name (e.g., `phi-3-mini`)
   - Configure tokens-per-minute quota and click **Deploy**
4. **Get the endpoint and API key**:
   - From the deployment page, copy the **Target URI** (endpoint)
   - Copy the **API Key** from **Keys and Endpoint** in the resource
5. **Store credentials** in a key file (e.g., `slm-azure-keys.md`):
   - Line 1: Endpoint URL (e.g., `https://<resource>.openai.azure.com/`)
   - Line 2: API Key
   - Line 3: Deployment name (e.g., `phi-3-mini`)

In [ ]:
from openai import AzureOpenAI

# --- SLM Configuration ---
OCR_OUTPUT_DIR = os.path.join("C:", os.sep, "Users", "shsenthi", "Downloads", "ocr_output")
LLM_OUTPUT_DIR = os.path.join("C:", os.sep, "Users", "shsenthi", "Downloads", "llm_json")
SLM_KEY_FILE = os.path.join("C:", os.sep, "Users", "shsenthi", "downloads", "slm-azure-keys.md")

# Read SLM endpoint, API key, and deployment name
with open(SLM_KEY_FILE, "r") as f:
    slm_lines = [line.strip() for line in f.readlines() if line.strip()]
    slm_endpoint = slm_lines[0]    # e.g., https://<resource>.openai.azure.com/
    slm_api_key = slm_lines[1]     # API key
    slm_deployment = slm_lines[2]  # e.g., phi-3-mini

print(f"SLM Endpoint   : {slm_endpoint}")
print(f"SLM API Key    : {slm_api_key[:6]}{'*' * (len(slm_api_key) - 6)}")
print(f"SLM Deployment : {slm_deployment}")

# Initialize Azure OpenAI client
slm_client = AzureOpenAI(
    azure_endpoint=slm_endpoint,
    api_key=slm_api_key,
    api_version="2024-06-01"
)
# --- Test the SLM service ---
print("\nTesting SLM service connectivity...")
try:
    test_response = slm_client.chat.completions.create(
        model=slm_deployment,
        messages=[
            {"role": "system", "content": "You are a helpful assistant. Respond with valid JSON only."},
            {"role": "user", "content": 'Extract entity name value pairs from: "Invoice #1234, Amount: $500, Date: 2026-01-15, Vendor: Contoso Ltd". Return only valid JSON.'}
        ],
        temperature=0.0,
        max_tokens=256
    )
    test_output = test_response.choices[0].message.content.strip()
    print(f"SLM service validated successfully!")
    print(f"  Model: {test_response.model}")
    print(f"  Usage: {test_response.usage.prompt_tokens} prompt + {test_response.usage.completion_tokens} completion tokens")
    print(f"  Test response:\n{test_output}")
except Exception as e:
    raise RuntimeError(f"SLM service test failed. Check endpoint, API key, and deployment name.\nError: {e}")

# Ensure LLM output directory exists
os.makedirs(LLM_OUTPUT_DIR, exist_ok=True)


In [ ]:

# Process each JSON file from OCR output
json_files = [f for f in os.listdir(OCR_OUTPUT_DIR) if f.lower().endswith(".json")]
print(f"\nFound {len(json_files)} JSON file(s) in OCR output directory.\n")

PROMPT = "Extract entity name value pairs from the following text and convert them strictly into JSON. Return only valid JSON, no explanation or markdown."

for json_filename in json_files:
    json_path = os.path.join(OCR_OUTPUT_DIR, json_filename)
    print(f"Processing: {json_filename} ...")

    # Read the OCR JSON and extract the content field
    with open(json_path, "r", encoding="utf-8") as f:
        ocr_data = json.load(f)

    content_text = ocr_data.get("content", "")
    if not content_text:
        print(f"  -> Skipped: no 'content' field found.")
        continue
    print (f"  -> Extracted content length: {len(content_text)} characters")
    print (f"  -> Sample content:\n{content_text[:200]}{'...' if len(content_text) > 200 else ''}")
    # Call the SLM with the OCR content
    try:
        response = slm_client.chat.completions.create(
            model=slm_deployment,
            messages=[
                {"role": "system", "content": "You are a structured data extraction assistant. Always respond with valid JSON only."},
                {"role": "user", "content": f"{PROMPT}\n\n{content_text}"}
            ],
            temperature=0.0,
            max_tokens=4096
        )

        raw_output = response.choices[0].message.content.strip()

        # Parse response as JSON, cleaning markdown fences if present
        cleaned = raw_output
        if cleaned.startswith("```json"):
            cleaned = cleaned[7:]
        if cleaned.startswith("```"):
            cleaned = cleaned[3:]
        if cleaned.endswith("```"):
            cleaned = cleaned[:-3]
        cleaned = cleaned.strip()

        entity_json = json.loads(cleaned)

    except json.JSONDecodeError:
        print(f"  -> Warning: SLM response was not valid JSON. Saving raw output.")
        entity_json = {"raw_response": raw_output}
    except Exception as e:
        print(f"  -> Error calling SLM: {e}")
        continue

    # Save the entity JSON output with same filename
    output_path = os.path.join(LLM_OUTPUT_DIR, json_filename)
    with open(output_path, "w", encoding="utf-8") as out_f:
        json.dump(entity_json, out_f, indent=2, ensure_ascii=False)

    print(f"  -> Saved: {json_filename}")

print("\nDone. All entity extractions complete.")